In [1]:
# Cell 1: Import thư viện cơ bản
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("CELL 1: Đã nạp NumPy thành công. Version:", np.__version__)

CELL 1: Đã nạp NumPy thành công. Version: 2.4.6


In [2]:
# Cell 2: Chứng minh KHÔNG có phi tuyến → mạng sâu suy biến thành 1 tầng

def affine(W, b, x):
    return W @ x + b

def relu(z):    return np.maximum(0, z)
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))
def softmax(z):
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()

# Hai tầng affine liên tiếp
W1 = np.random.randn(4, 3); b1 = np.random.randn(4)
W2 = np.random.randn(2, 4); b2 = np.random.randn(2)
x  = np.random.randn(3)

h = affine(W1, b1, x)
y_two_layers = affine(W2, b2, h)

# Gộp thành 1 tầng affine duy nhất
W_eff = W2 @ W1
b_eff = W2 @ b1 + b2
y_one_layer = W_eff @ x + b_eff

print("y_two_layers =", y_two_layers)
print("y_one_layer  =", y_one_layer)
print("Sai lệch max =", np.abs(y_two_layers - y_one_layer).max())
print("=> Không có phi tuyến, 2 tầng = 1 tầng.")

y_two_layers = [3.32645648 1.13682022]
y_one_layer  = [3.32645648 1.13682022]
Sai lệch max = 4.440892098500626e-16
=> Không có phi tuyến, 2 tầng = 1 tầng.


In [3]:
# Cell 3: Conv2D đơn kênh và đa kênh

def conv2d_manual(X, K, stride=1, padding=0, bias=0.0):
    if padding > 0:
        X = np.pad(X, padding, mode='constant')
    H, W = X.shape
    kh, kw = K.shape
    out_h = (H - kh) // stride + 1
    out_w = (W - kw) // stride + 1
    Y = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            window = X[i*stride:i*stride+kh, j*stride:j*stride+kw]
            Y[i, j] = np.sum(window * K) + bias
    return Y

def conv2d_multichannel(X, K, stride=1, padding=0, bias=None):
    C_in, H, W = X.shape
    C_out, _, kh, kw = K.shape
    if padding > 0:
        X = np.pad(X, ((0,0),(padding,padding),(padding,padding)), mode='constant')
    H, W = X.shape[1], X.shape[2]
    out_h = (H - kh) // stride + 1
    out_w = (W - kw) // stride + 1
    if bias is None: bias = np.zeros(C_out)
    Y = np.zeros((C_out, out_h, out_w))
    for f in range(C_out):
        for i in range(out_h):
            for j in range(out_w):
                window = X[:, i*stride:i*stride+kh, j*stride:j*stride+kw]
                Y[f, i, j] = np.sum(window * K[f]) + bias[f]
    return Y

# Ví dụ kernel Sobel dọc
X = np.array([[1,2,0,1,3],[0,1,3,2,1],[2,1,0,1,0],[1,0,2,3,1],[0,2,1,0,2]], dtype=float)
K_sobel = np.array([[1,0,-1],[1,0,-1],[1,0,-1]], dtype=float)
print("Conv2D với Sobel dọc:\n", conv2d_manual(X, K_sobel, padding=0))

Conv2D với Sobel dọc:
 [[ 0.  0. -1.]
 [-2. -4.  3.]
 [ 0. -1.  0.]]


In [4]:
# Cell 4: Công thức kích thước đầu ra và RF

def output_size(H, K, P=0, S=1, D=1):
    return (H + 2*P - D*(K-1) - 1) // S + 1

def receptive_field(kernels, strides):
    rf, jump = 1, 1
    for k, s in zip(kernels, strides):
        rf   += (k - 1) * jump
        jump *= s
    return rf

print("Conv 3x3, P=1, S=1 trên 28x28 :", output_size(28, 3, 1, 1))
print("Conv 3x3, P=0, S=2 trên 28x28 :", output_size(28, 3, 0, 2))
print("4 lớp Conv 3x3 liên tiếp, RF  :", receptive_field([3,3,3,3], [1,1,1,1]))
print("Conv3-Pool2-Conv3-Pool2, RF    :", receptive_field([3,3,2,3,2], [1,1,2,1,2]))

Conv 3x3, P=1, S=1 trên 28x28 : 28
Conv 3x3, P=0, S=2 trên 28x28 : 13
4 lớp Conv 3x3 liên tiếp, RF  : 9
Conv3-Pool2-Conv3-Pool2, RF    : 12


In [5]:
# Cell 5: Các toán tử bổ trợ

def max_pool2d(X, size=2, stride=2):
    H, W = X.shape
    out_h = (H - size) // stride + 1
    out_w = (W - size) // stride + 1
    Y = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            Y[i, j] = X[i*stride:i*stride+size, j*stride:j*stride+size].max()
    return Y

def avg_pool2d(X, size=2, stride=2):
    H, W = X.shape
    out_h = (H - size) // stride + 1
    out_w = (W - size) // stride + 1
    Y = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            Y[i, j] = X[i*stride:i*stride+size, j*stride:j*stride+size].mean()
    return Y

def global_avg_pool(X): return X.mean(axis=(1, 2))
def flatten(x):         return x.reshape(-1)

def batchnorm_inference(x, gamma, beta, running_mean, running_var, eps=1e-5):
    x_hat = (x - running_mean) / np.sqrt(running_var + eps)
    return gamma * x_hat + beta

def dense(x, W, b, activation=None):
    z = W @ x + b
    if activation == 'relu':    return relu(z)
    if activation == 'sigmoid': return sigmoid(z)
    if activation == 'softmax': return softmax(z)
    return z

# Demo
P = np.array([[1,3,2,1],[4,6,5,0],[3,1,1,2],[0,2,7,4]], dtype=float)
print("Max Pooling:\n", max_pool2d(P))
print("Avg Pooling:\n", avg_pool2d(P))

Max Pooling:
 [[6. 5.]
 [3. 7.]]
Avg Pooling:
 [[3.5 2. ]
 [1.5 3.5]]


In [6]:
# Cell 6: Tích chập 1D trên vector đặc trưng

def conv1d_manual(x, K, stride=1, padding=0):
    if padding > 0:
        x = np.pad(x, padding, mode='constant')
    L = len(x); k = len(K)
    out = (L - k) // stride + 1
    Y = np.zeros(out)
    for i in range(out):
        Y[i] = np.sum(x[i*stride:i*stride+k] * K)
    return Y

vec  = np.array([25.0, 0.0, 1.0, 27.3, 6.6, 140.0])
kern = np.array([0.5, -0.3, 0.2])
print("Conv1D trên 6 đặc trưng:", conv1d_manual(vec, kern, padding=0))

Conv1D trên 6 đặc trưng: [12.7   5.16 -6.37 39.67]


In [7]:
# Cell 7: Hàm hợp hoàn chỉnh (Conv → ReLU → Pool → Conv → ReLU → Pool → Flatten → Dense)

def forward_cnn_basic(x, params):
    x = conv2d_manual(x, params['K1'], padding=1, bias=params['b1'])
    x = relu(x); x = max_pool2d(x, 2, 2)
    x = conv2d_manual(x, params['K2'], padding=1, bias=params['b2'])
    x = relu(x); x = max_pool2d(x, 2, 2)
    x = flatten(x)
    out = dense(x, params['W_out'], params['b_out'], activation='softmax')
    return out

# Ví dụ chạy thử
demo_x = np.random.randn(8, 8)
demo_params = {
    'K1': np.random.randn(3, 3)*0.1, 'b1': 0.0,
    'K2': np.random.randn(3, 3)*0.1, 'b2': 0.0,
    'W_out': np.random.randn(3, 4)*0.1, 'b_out': np.zeros(3)
}
print("Output softmax:", forward_cnn_basic(demo_x, demo_params))

Output softmax: [0.33409687 0.33182497 0.33407817]


In [8]:
# Cell 8: Bảng đối chiếu hàm tự viết ↔ lớp PyTorch/Keras

import pandas as pd
mapping = [
    ("conv2d_manual",       "nn.Conv2d / layers.Conv2D"),
    ("relu",                "nn.ReLU / layers.Activation('relu')"),
    ("max_pool2d",          "nn.MaxPool2d / layers.MaxPooling2D"),
    ("avg_pool2d",          "nn.AvgPool2d / layers.AveragePooling2D"),
    ("global_avg_pool",     "nn.AdaptiveAvgPool2d(1)"),
    ("batchnorm_inference", "nn.BatchNorm2d / layers.BatchNormalization"),
    ("dense",               "nn.Linear / layers.Dense"),
    ("softmax",             "nn.Softmax / layers.Softmax"),
    ("conv1d_manual",       "nn.Conv1d / layers.Conv1D"),
]
df = pd.DataFrame(mapping, columns=["Hàm NumPy tự viết", "Lớp Keras/PyTorch tương ứng"])
print(df.to_string(index=False))

  Hàm NumPy tự viết                Lớp Keras/PyTorch tương ứng
      conv2d_manual                  nn.Conv2d / layers.Conv2D
               relu        nn.ReLU / layers.Activation('relu')
         max_pool2d         nn.MaxPool2d / layers.MaxPooling2D
         avg_pool2d     nn.AvgPool2d / layers.AveragePooling2D
    global_avg_pool                    nn.AdaptiveAvgPool2d(1)
batchnorm_inference nn.BatchNorm2d / layers.BatchNormalization
              dense                   nn.Linear / layers.Dense
            softmax                nn.Softmax / layers.Softmax
      conv1d_manual                  nn.Conv1d / layers.Conv1D
